# 感知受限场景生成 — 结果分析与可视化

合并多个 seed 生成的危险场景库，输出统计量、分布并可视化，便于分析所有生成危险场景的整体分布。

数据来源：`ppo_logs_gpu*/vae-ppo_vehicle_trajectories_*.h5`（每张卡一个独立 seed 的训练结果）。

## 数据 schema

- 每个 `episode_*` 组，attrs：`collision / episode_length / episode_reward`
  - `trajectories` `(T, 5, 4)`：`[纵向位置, 横向位置 lane_pos, 纵向速度, 0]`，车顺序 `[ego, adversary, bg2, bg3, bg4]`
  - `perception_data` `(T, 5, 13)`：`[感知值(4) | 真实值(4) | 误差 delta(4) | 距离(1)]`
    - 感知值：`[perceived_pos, perceived_lane_pos, perceived_speed, perceived_vy]`
    - 真实值：`[true_pos, true_lane_pos, true_speed, true_vy]`
    - 误差：`[dx, dy, dvx, dvy]`
    - 距离：`dist_to_ego`

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import h5py
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['axes.grid'] = True

# ===== 配置（按需修改）=====
DATA_DIR = "/workspace/asw-shared/dlp/training_tasks/aoh6szh/n260827-174817-scegen-ppo"
H5_GLOB = os.path.join(DATA_DIR, "ppo_logs_gpu*/vae-ppo_vehicle_trajectories_*.h5")

# ===== 常量（必须与 sce_gen.py 一致）=====
CAR_LENGTH = 5.0
LANE_WIDTH = 4.0
SAME_LANE_THRESH = 2.2
TTC_LOW = 1.0
TTC_HIGH = 4.0
TIME_STEP = 0.2

# 感知噪声拟合系数（论文 Table III），用于叠加拟合曲线验证
PERC_COEFFS = {
    'dx':  {'mu': (-0.00691, -0.00013, -4.64218e-07), 'sigma': (0.03711, 0.00511, -5.68800e-05)},
    'dy':  {'mu': (-0.00571,  0.00039, -4.90203e-08), 'sigma': (0.06703, 0.00593, -7.43626e-05)},
    'dvx': {'mu': (-0.00369,  0.00054, -6.30793e-06), 'sigma': (0.06579, 0.00693, -7.20104e-05)},
    'dvy': {'mu': ( 0.00864, -0.00323,  6.23654e-05), 'sigma': (0.31881, 0.00069,  5.11978e-06)},
}

files = sorted(glob.glob(H5_GLOB))
print(f"找到 {len(files)} 个 h5 文件")
for f in files:
    print("  ", f)

In [ ]:
def min_ttc_of_trajectory(traj):
    """traj: (T, N, 4) = [pos, lane_pos, speed, 0]，car0=ego。返回该 episode 的最小 TTC。"""
    ego = traj[:, 0, :]
    best = np.inf
    with np.errstate(divide='ignore', invalid='ignore'):
        for i in range(1, traj.shape[1]):
            npc = traj[:, i, :]
            dx = npc[:, 0] - ego[:, 0]
            rel = ego[:, 2] - npc[:, 2]
            lane_ok = np.abs(ego[:, 1] - npc[:, 1]) < SAME_LANE_THRESH
            for m, ttc in ((lane_ok & (dx > 0) & (rel > 0), (dx - CAR_LENGTH) / rel),
                           (lane_ok & (dx < 0) & (rel < 0), (-dx - CAR_LENGTH) / (-rel))):
                if m.any():
                    v = ttc[m]
                    v = v[v > 0]
                    if v.size:
                        best = min(best, v.min())
    return best if np.isfinite(best) else 100.0


def poly2(c, d):
    a0, a1, a2 = c
    return a0 + a1 * d + a2 * d ** 2


def ttc_matrix(ego, others, perceived):
    """逐 timestep 计算 ego 相对所有背景车的最小 TTC。

    ego: (T, 1, 13)；others: (T, M, 13)。
    perceived=True 用感知列 (0,1,2)，False 用真实列 (4,5,6)。返回 (T,) 数组。
    """
    pos, lane, spd = (0, 1, 2) if perceived else (4, 5, 6)
    ego_pos = ego[:, :, pos]
    ego_lane = ego[:, :, lane]
    ego_spd = ego[:, :, spd]
    o_pos = others[:, :, pos]
    o_lane = others[:, :, lane]
    o_spd = others[:, :, spd]
    dx = o_pos - ego_pos
    rel = ego_spd - o_spd
    lane_ok = np.abs(ego_lane - o_lane) < SAME_LANE_THRESH
    with np.errstate(divide='ignore', invalid='ignore'):
        fwd = (dx - CAR_LENGTH) / rel
        rear = (-dx - CAR_LENGTH) / (-rel)
    fwd = np.where(lane_ok & (dx > 0) & (rel > 0) & (fwd > 0), fwd, np.inf)
    rear = np.where(lane_ok & (dx < 0) & (rel < 0) & (rear > 0), rear, np.inf)
    return np.minimum(fwd, rear).min(axis=1)

In [ ]:
rows = []
for path in files:
    run = os.path.basename(os.path.dirname(path))  # ppo_logs_gpu0_...
    with h5py.File(path, 'r') as f:
        for k in f.keys():
            g = f[k]
            attrs = dict(g.attrs)
            traj = g['trajectories'][:]  # (T,5,4)
            rows.append(dict(
                run=run,
                episode=int(k.split('_')[1]),
                length=traj.shape[0],
                collision=bool(attrs.get('collision', False)),
                reward=float(attrs.get('episode_reward', 0.0)),
                min_ttc=min_ttc_of_trajectory(traj),
                path=path,
            ))
meta = pd.DataFrame(rows)
print("总场景数:", len(meta))
meta.head()

In [ ]:
print("===== 总体统计 =====")
print(f"场景总数        : {len(meta)}")
print(f"run(seed) 数    : {meta['run'].nunique()}")
print(f"碰撞场景        : {int(meta['collision'].sum())}  ({100*meta['collision'].mean():.2f}%)")
print(f"危险场景 TTC<{TTC_HIGH:.0f}: {(meta['min_ttc']<TTC_HIGH).sum()}  ({100*(meta['min_ttc']<TTC_HIGH).mean():.2f}%)")
print(f"严重危险 TTC<{TTC_LOW:.0f}: {(meta['min_ttc']<TTC_LOW).sum()}  ({100*(meta['min_ttc']<TTC_LOW).mean():.2f}%)")
print()
print(meta[['length', 'reward', 'min_ttc']].describe().round(3))

In [ ]:
by_run = meta.groupby('run').agg(
    episodes=('episode', 'count'),
    collision_rate=('collision', 'mean'),
    dangerous_rate=('min_ttc', lambda s: (s < TTC_HIGH).mean()),
    mean_min_ttc=('min_ttc', 'mean'),
    mean_length=('length', 'mean'),
    mean_reward=('reward', 'mean'),
).reset_index()
by_run.round(4)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].hist(meta['length'], bins=40, color='tab:blue', alpha=0.7)
axes[0, 0].set_title('Episode length (steps)')
axes[0, 0].set_xlabel('steps')

ttc = meta['min_ttc'].clip(lower=0.05)
axes[0, 1].hist(ttc, bins=60, color='tab:orange', alpha=0.7)
axes[0, 1].set_xscale('log')
axes[0, 1].axvline(TTC_LOW, color='red', ls='--', label='TTC_LOW=1')
axes[0, 1].axvline(TTC_HIGH, color='green', ls='--', label='TTC_HIGH=4')
axes[0, 1].set_title('Min TTC (log scale)')
axes[0, 1].legend()

axes[0, 2].hist(meta['reward'], bins=50, color='tab:green', alpha=0.7)
axes[0, 2].set_title('Episode reward')

axes[1, 0].bar(by_run['run'], by_run['collision_rate'] * 100, color='tab:red', alpha=0.7)
axes[1, 0].set_title('Collision rate per run (%)')
axes[1, 0].tick_params(axis='x', rotation=45)

axes[1, 1].bar(by_run['run'], by_run['dangerous_rate'] * 100, color='tab:purple', alpha=0.7)
axes[1, 1].set_title(f'Dangerous rate (TTC<{TTC_HIGH:.0f}) per run (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

axes[1, 2].boxplot([meta[meta['run'] == r]['min_ttc'].clip(lower=0.05) for r in by_run['run']],
                   labels=by_run['run'])
axes[1, 2].set_yscale('log')
axes[1, 2].set_title('Min TTC per run (log)')
axes[1, 2].tick_params(axis='x', rotation=45)

fig.tight_layout()
plt.show()

## 感知误差分析

从 `perception_data` 里提取所有非自车在全部 timestep 上的 `(dist, dx, dy, dvx, dvy)`，分析误差分布与「误差随距离」的变化，并叠加论文拟合的 `mu(d)` / `sigma(d)` 曲线做验证。

In [ ]:
def collect_perception_errors(files):
    dists, dxs, dys, dvxs, dvys = [], [], [], [], []
    for path in files:
        with h5py.File(path, 'r') as f:
            for k in f.keys():
                p = f[k]['perception_data'][:]  # (T,5,13)
                npe = p[:, 1:, :]  # 非自车
                dists.append(npe[:, :, 12].ravel())
                dxs.append(npe[:, :, 8].ravel())
                dys.append(npe[:, :, 9].ravel())
                dvxs.append(npe[:, :, 10].ravel())
                dvys.append(npe[:, :, 11].ravel())
    return tuple(np.concatenate(x) for x in (dists, dxs, dys, dvxs, dvys))

dist, dx, dy, dvx, dvy = collect_perception_errors(files)
print(f"感知误差样本数: {len(dist):,}")
print(pd.DataFrame({'dx': dx, 'dy': dy, 'dvx': dvx, 'dvy': dvy}).describe().round(4))

In [ ]:
errors = {'dx': dx, 'dy': dy, 'dvx': dvx, 'dvy': dvy}

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (name, e) in zip(axes.ravel(), errors.items()):
    ax.hist(e, bins=80, density=True, alpha=0.7, color='tab:blue')
    ax.set_title(f'{name} 误差分布')
    ax.set_xlabel(name)
    ax.set_ylabel('density')
fig.tight_layout()
plt.show()

# 误差随距离变化 + 叠加 mu(d), mu±sigma(d)
d_grid = np.linspace(0, dist.max(), 200)
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (name, e) in zip(axes.ravel(), errors.items()):
    mu_c = PERC_COEFFS[name]['mu']
    sg_c = PERC_COEFFS[name]['sigma']
    ax.hexbin(dist, e, gridsize=60, bins='log', cmap='Blues', mincnt=1)
    mu_d = poly2(mu_c, d_grid)
    sg_d = np.abs(poly2(sg_c, d_grid))
    ax.plot(d_grid, mu_d, 'r-', lw=2, label='mu(d)')
    ax.plot(d_grid, mu_d + sg_d, 'r--', lw=1, label='mu +/- sigma(d)')
    ax.plot(d_grid, mu_d - sg_d, 'r--', lw=1)
    ax.set_xlabel('距离 d (m)')
    ax.set_ylabel(name)
    ax.set_title(f'{name} vs 距离')
    ax.legend()
fig.tight_layout()
plt.show()

## 感知导致的风险低估（ERD-lite）

论文的 ERD 用前向 IDM 仿真计算 `Delta = E[R_obs] - E[R_true]`。这里用**逐时刻的瞬时 TTC** 做近似：用 ego 的感知视图算感知 TTC、用真实视图算真实 TTC，`risk = 1/TTC`，`risk_true - risk_perc > 0` 表示自车**低估了危险**（以为自己更安全）。

In [ ]:
def collect_risk_underestimation(files):
    all_true, all_perc, all_under = [], [], []
    for path in files:
        with h5py.File(path, 'r') as f:
            for k in f.keys():
                p = f[k]['perception_data'][:]
                ego = p[:, 0:1, :]
                others = p[:, 1:, :]
                t_true = ttc_matrix(ego, others, perceived=False)
                t_perc = ttc_matrix(ego, others, perceived=True)
                r_true = 1.0 / np.clip(t_true, 0.1, None)
                r_perc = 1.0 / np.clip(t_perc, 0.1, None)
                all_true.append(t_true)
                all_perc.append(t_perc)
                all_under.append(r_true - r_perc)
    return (np.concatenate(x) for x in (all_true, all_perc, all_under))

ttc_true, ttc_perc, under = collect_risk_underestimation(files)
print(f"timestep 样本数: {len(ttc_true):,}")

danger = ttc_true < TTC_HIGH
print(f"\n真实 TTC<{TTC_HIGH:.0f} 的时刻占比: {100*danger.mean():.2f}%")
print(f"  其中感知 TTC > 真实 TTC（低估危险）占比: {100*(ttc_perc[danger] > ttc_true[danger]).mean():.2f}%")
print(f"  低估幅度均值: {under[danger & (under > 0)].mean():.3f} (1/TTC 单位)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(under, bins=80, color='tab:red', alpha=0.7)
axes[0].axvline(0, color='k', ls='--')
axes[0].set_title('风险低估 = risk_true - risk_perc（正向 = 危险低估）')
axes[0].set_xlabel('risk_true - risk_perc')

m = (ttc_true < 20) & (ttc_perc < 20)
axes[1].hexbin(ttc_true[m], ttc_perc[m], gridsize=60, bins='log', cmap='Greens', mincnt=1)
axes[1].plot([0, 20], [0, 20], 'r--', label='perceived = true')
axes[1].set_xlabel('真实 TTC (s)')
axes[1].set_ylabel('感知 TTC (s)')
axes[1].set_title('感知 vs 真实 TTC（对角线上方 = 低估危险）')
axes[1].legend()
plt.tight_layout()
plt.show()

## 样例场景可视化

各挑一个碰撞场景和一个最小 TTC 场景，画出各车纵向位置与横向位置随时间的变化。

In [ ]:
def plot_episode(path, episode_idx):
    with h5py.File(path, 'r') as f:
        g = f[f"episode_{episode_idx}"]
        traj = g['trajectories'][:]
        attrs = dict(g.attrs)
    T = traj.shape[0]
    t = np.arange(T) * TIME_STEP
    names = ['ego', 'adversary', 'bg2', 'bg3', 'bg4']
    colors = ['black', 'red', 'tab:gray', 'tab:gray', 'tab:gray']
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for i in range(traj.shape[1]):
        axes[0].plot(t, traj[:, i, 0], color=colors[i], label=names[i])
        axes[1].plot(t, traj[:, i, 1], color=colors[i], label=names[i])
    axes[0].set_xlabel('时间 (s)'); axes[0].set_ylabel('纵向位置 (m)')
    axes[0].set_title(f"episode_{episode_idx} 纵向位置 (collision={attrs.get('collision')})")
    axes[1].set_xlabel('时间 (s)'); axes[1].set_ylabel('横向位置 lane_pos (m)')
    axes[1].set_title('横向位置 / 换道')
    for ax in axes:
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

coll = meta[meta['collision']]
if len(coll):
    r = coll.iloc[0]
    plot_episode(r['path'], r['episode'])

near = meta[meta['min_ttc'] == meta['min_ttc'].min()]
if len(near):
    r = near.iloc[0]
    plot_episode(r['path'], r['episode'])